# 06 · Seen vs unseen users: plain SMLP4Rec vs hybrid (SMLP4Rec + prior + sameDest)

Test-only notebook: **no training, no tuning**. It reloads notebook 01's best checkpoint and notebook 04's chosen hybrid weights (`results/week3_implementation/smlprec_expedia_hybrid_samedest.json`), rescores valid and test, and splits the events by whether the user is present in the training set.

**Seen vs unseen user**

- **Seen**: the user has at least one row in the RecBole **train split** (a train target: a booking in the train window that is not the user's first booking). The model was trained on next-booking targets of this user.
- **Unseen (cold-start)**: no train row. Two kinds of events:
  - **unseen, L ≥ 1**: the user has earlier bookings, but all of them (except possibly a single first booking) fall after the train window, so the history the model reads at test time was never a training label.
  - **unseen, L = 0**: the user's first booking (no history at all). These are the "cold rows" of notebooks 03/04.
- Every L = 0 row is unseen by construction (asserted below).

**Approaches (same as notebooks 03/04, fixed weights)**

| | warm rows (L ≥ 1) | cold rows (L = 0) |
|---|---|---|
| plain SMLP4Rec | model softmax over the 100 clusters | global popularity (the model needs ≥ 1 past booking) |
| hybrid | `log q_SMLP4Rec + w_p log q_prior + w_s log q_sameDest`, alpha and per-cell weights from notebook 04 (cells = known vs new destination) | destination prior only |

Metrics: Recall@K and NDCG@K (K = 5, 10, 20), full ranking over the 100 clusters; paired bootstrap 95 % intervals for hybrid minus plain (Recall@5, NDCG@5).

Kernel: `smlp4rec`. Needs `data/interim/recbole/expedia_dest/expedia_dest.inter` and the raw `train.csv` (prior tables).

In [1]:
import functools
import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll issue)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import summarize, target_rank
from nbp.hybrid.fusion import same_dest_distribution, smoothed_log
from nbp.hybrid.same_dest import same_dest_scores
from src.models.smlprec import SMLPREC

logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
RAW = ROOT / "data" / "raw" / "hospitality" / "expedia" / "train.csv"
INTER_DEST = WORK / "expedia_dest" / "expedia_dest.inter"
REF_DIR = ROOT / "results" / "week3_implementation"
RUN_DIR = max((ROOT / "experiments").glob("*_expedia_smlp4rec_week3-repro"))

N_CLUSTERS = 100
M_SMOOTH = 5
EPS = 1e-6  # prior floor for cold rows (notebooks 03/04)
TIME_MARGIN = 128  # seconds; float32 timestamp resolution near 1.4e9
RECENCY = 0.7

HYB_REF = json.loads((REF_DIR / "smlprec_expedia_hybrid_samedest.json").read_text(encoding="utf-8"))
SCHEME = HYB_REF["cohorts"]["chosen_fewest_cells_within_one_se"]
assert SCHEME == "known vs new destination (2)", SCHEME
FINAL_W = {int(k): tuple(v) for k, v in HYB_REF["cohorts"]["final_weights_per_cell"].items()}
ALPHA = HYB_REF["selection"]["one_se_pick"][0]
assert all(w[0] == ALPHA for w in FINAL_W.values())
HYB_NAME = "SMLP4Rec + prior + sameDest (this notebook)"
print("run dir:", RUN_DIR)
print("hybrid weights from notebook 04: alpha", ALPHA, "| cell 0 = new destination, 1 = known destination:", FINAL_W)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-05_expedia_smlp4rec_week3-repro
hybrid weights from notebook 04: alpha 0.2 | cell 0 = new destination, 1 = known destination: {0: (0.2, 2.0, 0.0), 1: (0.2, 1.25, 1.0)}


## 1. Data, prior and score components (same code as notebook 04)

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_dest",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "srch_destination_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])
t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
tok2id = dataset.field2token_id[dataset.iid_field]
uid_of_token = dataset.field2token_id[dataset.uid_field]
cluster_ids = np.array([tok2id[str(k)] for k in range(N_CLUSTERS)])  # model column of each raw cluster
col_of_model_id = {int(m): k for k, m in enumerate(cluster_ids)}
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
TIME = config["TIME_FIELD"]
UID = config["USER_ID_FIELD"]


def split_arrays(loader) -> dict:
    d = loader.dataset.inter_feat
    seq, length = d[ITEM_SEQ].numpy(), d[ITEM_SEQ_LEN].numpy()
    y_model = d[config["ITEM_ID_FIELD"]].numpy()
    q_dest = d["srch_destination_id"].numpy().astype(np.int64)
    h_dest = d["srch_destination_id" + config["LIST_SUFFIX"]].numpy().astype(np.int64)
    pos = np.arange(seq.shape[1])[None, :] < length[:, None]
    return {
        "seq": seq,
        "seq_t": d[ITEM_SEQ],
        "length": length,
        "length_t": d[ITEM_SEQ_LEN],
        "y": np.array([col_of_model_id[int(t)] for t in y_model]),  # raw cluster index 0..99
        "user": d[UID].numpy(),
        "q_dest": q_dest,
        "h_dest": h_dest,
        "q_seen": ((h_dest == q_dest[:, None]) & pos).any(1),
    }


arr = {"valid": split_arrays(valid_data), "test": split_arrays(test_data)}
train_users = np.unique(train_data.dataset.inter_feat[UID].numpy())
print({sp: len(a["y"]) for sp, a in arr.items()}, "| users with a train row:", len(train_users), f"| {time.time() - t0:.0f}s")

{'valid': 218670, 'test': 218670} | users with a train row: 419324 | 157s


In [3]:
import pandas as pd  # after torch

t_train_max = float(train_data.dataset.inter_feat[TIME].max())
t_valid_max = float(valid_data.dataset.inter_feat[TIME].max())
cut = t_train_max - TIME_MARGIN


def build_prior_tables(cut_seconds: float) -> dict:
    """Cluster count tables from bookings with unix time < cut_seconds (one chunked pass over train.csv)."""
    cols = ["date_time", "is_booking", "srch_destination_id", "hotel_market", "hotel_cluster"]
    dtypes = {c: "int32" for c in cols if c != "date_time"}
    parts = []
    for ch in pd.read_csv(RAW, usecols=cols, chunksize=2_000_000, dtype=dtypes):
        ch = ch[ch.is_booking == 1]
        ts = (pd.to_datetime(ch["date_time"]) - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)
        parts.append(ch.loc[ts.values < cut_seconds, ["srch_destination_id", "hotel_market", "hotel_cluster"]])
    b = pd.concat(parts, ignore_index=True)

    def table(key):
        t = b.groupby([key, "hotel_cluster"]).size().unstack(fill_value=0)
        return t.reindex(columns=range(N_CLUSTERS), fill_value=0)

    dest, mkt = table("srch_destination_id"), table("hotel_market")
    dm = b.groupby(["srch_destination_id", "hotel_market"]).size().reset_index(name="n")
    modal = dm.sort_values("n", ascending=False).drop_duplicates("srch_destination_id")
    glob = np.bincount(b["hotel_cluster"].values, minlength=N_CLUSTERS).astype(np.float64)
    return {
        "n_bookings": len(b),
        "dest": {int(d): r.astype(np.float64) for d, r in zip(dest.index, dest.values)},
        "market": {int(m): r.astype(np.float64) for m, r in zip(mkt.index, mkt.values)},
        "dest2mkt": dict(zip(modal["srch_destination_id"].astype(int), modal["hotel_market"].astype(int))),
        "global": glob / glob.sum(),
    }


def prior_vector(d: int) -> np.ndarray:
    """p(cluster | destination), shape (100,), raw cluster order."""
    row = tables["dest"].get(d)
    if row is None:
        return tables["global"]
    pm = tables["market"][tables["dest2mkt"][d]]
    pm = pm / pm.sum()
    return (row + M_SMOOTH * pm) / (row.sum() + M_SMOOTH)


def prior_prob(dests: np.ndarray) -> np.ndarray:
    """(n, 100) prior distribution of each query destination, raw cluster order."""
    uniq, inv = np.unique(dests, return_inverse=True)
    return np.stack([prior_vector(int(d)) for d in uniq]).astype(np.float32)[inv]


t1 = time.time()
tables = build_prior_tables(cut)
print(f"prior cut (unix s) {cut:.0f}; {tables['n_bookings']} bookings, {len(tables['dest'])} destinations; {time.time() - t1:.0f}s")

model = SMLPREC(config, dataset)
model.load_state_dict(torch.load(RUN_DIR / "best.pth")["state_dict"])


@torch.no_grad()
def model_prob(a: dict, batch: int = 8192) -> np.ndarray:
    """(n, 100) softmax of the model over the 100 clusters (padding column excluded), raw cluster order."""
    model.eval()
    out = []
    for i in range(0, len(a["y"]), batch):
        s = model.full_sort_predict(
            {model.ITEM_SEQ: a["seq_t"][i : i + batch], model.ITEM_SEQ_LEN: a["length_t"][i : i + batch]}
        )
        s[:, 0] = float("-inf")
        out.append(torch.softmax(s, dim=1).numpy()[:, cluster_ids])
    return np.concatenate(out)


t2 = time.time()
p_model, p_prior, sd = {}, {}, {}
for sp, a in arr.items():
    p_model[sp], p_prior[sp] = model_prob(a), prior_prob(a["q_dest"])
    sd[sp] = same_dest_scores(
        a["seq"], a["length"], a["h_dest"], a["q_dest"], dataset.item_num, base=RECENCY, normalize=True
    )[:, cluster_ids]
print(f"components ready in {time.time() - t2:.0f}s", {sp: p_model[sp].shape for sp in arr})

prior cut (unix s) 1412776576; 2463082 bookings, 34506 destinations; 74s
components ready in 19s {'valid': (218670, 100), 'test': (218670, 100)}


## 2. Cold rows (L = 0)

Each user's first booking, in the valid window `[last train target, last valid target)` and the test window `[last valid target, end]`, as in notebooks 03/04.

In [4]:
inter = pd.read_csv(INTER_DEST, sep="\t").sort_values(["user_id:token", "timestamp:float"], kind="stable")
first = inter.drop_duplicates("user_id:token", keep="first")
windows = {
    "valid": (first["timestamp:float"] >= t_train_max) & (first["timestamp:float"] < t_valid_max),
    "test": first["timestamp:float"] >= t_valid_max,
}
cold = {}
for sp, mask in windows.items():
    f = first[mask]
    cold[sp] = {
        "y": f["item_id:token"].astype(int).values,  # raw cluster index 0..99
        "dest": f["srch_destination_id:float"].values.astype(np.int64),
        "user": np.array([uid_of_token[str(u)] for u in f["user_id:token"]]),
    }
    assert not np.isin(cold[sp]["user"], train_users).any(), "a first booking user has a train row"
print({sp: len(c["y"]) for sp, c in cold.items()})

{'valid': 50803, 'test': 49443}


## 3. Seen vs unseen: how many users and events

Counts per split, over **all events** of the split (warm rows + cold rows). A user is counted once per split; "% of users" is over the distinct users who have at least one event in that split.

In [5]:
seen_rows = {sp: np.isin(arr[sp]["user"], train_users) for sp in arr}


def presence_stats(sp: str) -> dict:
    w_user, c_user, w_seen = arr[sp]["user"], cold[sp]["user"], seen_rows[sp]
    users = np.unique(np.concatenate([w_user, c_user]))
    seen_u = np.isin(users, train_users)
    n_rows = len(w_user) + len(c_user)
    unseen_warm_users = np.unique(w_user[~w_seen])
    return {
        "users": int(len(users)),
        "seen_users": int(seen_u.sum()),
        "unseen_users": int((~seen_u).sum()),
        "seen_users_pct": round(100 * float(seen_u.mean()), 2),
        "unseen_users_pct": round(100 * float((~seen_u).mean()), 2),
        "unseen_users_with_history_(L>=1)": int(len(unseen_warm_users)),
        "unseen_users_first_booking_only_(L=0)": int(len(np.setdiff1d(c_user, unseen_warm_users))),
        "events": int(n_rows),
        "seen_events": int(w_seen.sum()),
        "unseen_events": int((~w_seen).sum() + len(c_user)),
        "seen_events_pct": round(100 * float(w_seen.sum()) / n_rows, 2),
        "unseen_events_pct": round(100 * float((~w_seen).sum() + len(c_user)) / n_rows, 2),
        "unseen_events_L>=1": int((~w_seen).sum()),
        "unseen_events_L=0": int(len(c_user)),
        "mean_history_len_seen": round(float(arr[sp]["length"][w_seen].mean()), 3),
        "mean_history_len_unseen_L>=1": round(float(arr[sp]["length"][~w_seen].mean()), 3),
    }


presence = {sp: presence_stats(sp) for sp in ("valid", "test")}
for sp in presence:
    p = presence[sp]
    assert p["seen_users"] + p["unseen_users"] == p["users"]
    assert p["seen_events"] + p["unseen_events"] == p["events"]
pd.DataFrame(presence)

,valid,test
users,166748.000,165833.000
seen_users,88775.000,81146.000
unseen_users,77973.000,84687.000
seen_users_pct,53.240,48.930
unseen_users_pct,46.760,51.070
unseen_users_with_history_(L>=1),38989.000,48008.000
unseen_users_first_booking_only_(L=0),38984.000,36679.000
events,269473.000,268113.000
seen_events,161151.000,143025.000
unseen_events,108322.000,125088.000


## 4. Score both approaches (fixed weights) and check against the saved runs

- Plain SMLP4Rec on warm rows must reproduce notebook 01 (`metrics.json` of the run).
- Plain SMLP4Rec warm + global popularity cold must reproduce notebook 03's "plain SMLP4Rec on warm + global popularity on cold".
- Hybrid must reproduce notebook 04's test and valid numbers (warm and all events).

In [6]:
def rank(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    return target_rank(scores, y)


PLAIN = "plain SMLP4Rec"
HYB = "hybrid (SMLP4Rec + prior + sameDest)"
ranks = {sp: {} for sp in arr}
for sp, a in arr.items():
    cell = a["q_seen"].astype(int)  # scheme "known vs new destination": 1 = known
    wp = np.array([FINAL_W[c][1] for c in cell], np.float32)[:, None]
    ws = np.array([FINAL_W[c][2] for c in cell], np.float32)[:, None]
    M = smoothed_log(p_model[sp], ALPHA)
    P = smoothed_log(p_prior[sp], ALPHA)
    S = smoothed_log(same_dest_distribution(sd[sp]), ALPHA)
    c = cold[sp]
    glob_log = np.tile(np.log(tables["global"] + EPS).astype(np.float32), (len(c["y"]), 1))
    ranks[sp][PLAIN] = {"warm": rank(np.log(p_model[sp]), a["y"]), "cold": rank(glob_log, c["y"])}
    ranks[sp][HYB] = {
        "warm": rank(M + wp * P + ws * S, a["y"]),
        "cold": rank(np.log(prior_prob(c["dest"]) + EPS), c["y"]),
    }
    del M, P, S


def all_events(r: dict) -> np.ndarray:
    return np.concatenate([r["warm"], r["cold"]])


ref01 = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
LF = json.loads((REF_DIR / "smlprec_expedia_late_fusion.json").read_text(encoding="utf-8"))
checks = []
for sp in arr:
    got, want = summarize(ranks[sp][PLAIN]["warm"]), ref01[sp]
    checks.append((f"{sp} plain warm vs notebook 01", got, want))
    got = summarize(all_events(ranks[sp][PLAIN]))
    want = LF["combined_warm_plus_cold"][sp]["plain SMLP4Rec on warm + global popularity on cold"]
    checks.append((f"{sp} plain all events vs notebook 03", got, want))
    for scope in ("warm", "all_events"):
        r = ranks[sp][HYB]["warm"] if scope == "warm" else all_events(ranks[sp][HYB])
        checks.append((f"{sp} hybrid {scope} vs notebook 04", summarize(r), HYB_REF["benchmark"][sp][scope][HYB_NAME]))
MET = ["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
for name, got, want in checks:
    same = all(got[m] == want[m] for m in MET if m in want)
    print(f"{name:42s} {'OK' if same else 'MISMATCH'}  R@5 {got['recall@5']} vs {want['recall@5']}  N@5 {got['ndcg@5']} vs {want.get('ndcg@5')}")
    assert same, name

valid plain warm vs notebook 01            OK  R@5 0.3352 vs 0.3352  N@5 0.2429 vs 0.2429
valid plain all events vs notebook 03      OK  R@5 0.2956 vs 0.2956  N@5 0.2122 vs 0.2122
valid hybrid warm vs notebook 04           OK  R@5 0.6181 vs 0.6181  N@5 0.4591 vs 0.4591
valid hybrid all_events vs notebook 04     OK  R@5 0.6013 vs 0.6013  N@5 0.4407 vs 0.4407
test plain warm vs notebook 01             OK  R@5 0.3369 vs 0.3369  N@5 0.2439 vs 0.2439
test plain all events vs notebook 03       OK  R@5 0.299 vs 0.299  N@5 0.2139 vs 0.2139
test hybrid warm vs notebook 04            OK  R@5 0.6106 vs 0.6106  N@5 0.4539 vs 0.4539
test hybrid all_events vs notebook 04      OK  R@5 0.5925 vs 0.5925  N@5 0.4349 vs 0.4349


## 5. Results: plain SMLP4Rec vs hybrid on seen and unseen users

Subsets of each split's events (warm + cold):

- **seen users**: warm rows of users with a train row (no cold row can be seen).
- **unseen users**: warm rows of users without a train row + all cold rows. Broken down into **unseen, L ≥ 1** and **unseen, L = 0 (first booking)**, because the plain model falls back to global popularity on L = 0.
- **all events**: reference (equals notebooks 03/04).

In [7]:
SUBSETS = ["seen users", "unseen users", "unseen, L>=1", "unseen, L=0 (first booking)", "all events"]


def subset_ranks(sp: str, name: str) -> dict:
    r, s = ranks[sp][name], seen_rows[sp]
    return {
        "seen users": r["warm"][s],
        "unseen users": np.concatenate([r["warm"][~s], r["cold"]]),
        "unseen, L>=1": r["warm"][~s],
        "unseen, L=0 (first booking)": r["cold"],
        "all events": all_events(r),
    }


sub = {sp: {name: subset_ranks(sp, name) for name in (PLAIN, HYB)} for sp in arr}
results = {
    sp: {subset: {name: summarize(sub[sp][name][subset]) for name in (PLAIN, HYB)} for subset in SUBSETS}
    for sp in arr
}


def results_table(sp: str) -> pd.DataFrame:
    rows = {}
    for subset in SUBSETS:
        for name in (PLAIN, HYB):
            rows[(subset, name)] = results[sp][subset][name]
    return pd.DataFrame(rows).T[["n"] + MET].astype({"n": int})


print("TEST")
display(results_table("test"))
print("VALID")
display(results_table("valid"))

TEST


n  \
seen users                  plain SMLP4Rec                        143025   
                            hybrid (SMLP4Rec + prior + sameDest)  143025   
unseen users                plain SMLP4Rec                        125088   
                            hybrid (SMLP4Rec + prior + sameDest)  125088   
unseen, L>=1                plain SMLP4Rec                         75645   
                            hybrid (SMLP4Rec + prior + sameDest)   75645   
unseen, L=0 (first booking) plain SMLP4Rec                         49443   
                            hybrid (SMLP4Rec + prior + sameDest)   49443   
all events                  plain SMLP4Rec                        268113   
                            hybrid (SMLP4Rec + prior + sameDest)  268113   

                                                                  recall@5  \
seen users                  plain SMLP4Rec                          0.3406   
                            hybrid (SMLP4Rec + prior + sameDest)    0.6228   
unseen users                plain SMLP4Rec                          0.2516   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5578   
unseen, L>=1                plain SMLP4Rec                          0.3299   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5876   
unseen, L=0 (first booking) plain SMLP4Rec                          0.1317   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5121   
all events                  plain SMLP4Rec                          0.2990   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5925   

                                                                  ndcg@5  \
seen users                  plain SMLP4Rec                        0.2433   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4640   
unseen users                plain SMLP4Rec                        0.1804   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4015   
unseen, L>=1                plain SMLP4Rec                        0.2451   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4348   
unseen, L=0 (first booking) plain SMLP4Rec                        0.0813   
                            hybrid (SMLP4Rec + prior + sameDest)  0.3505   
all events                  plain SMLP4Rec                        0.2139   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4349   

                                                                  recall@10  \
seen users                  plain SMLP4Rec                           0.4707   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7833   
unseen users                plain SMLP4Rec                           0.3634   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7349   
unseen, L>=1                plain SMLP4Rec                           0.4526   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7567   
unseen, L=0 (first booking) plain SMLP4Rec                           0.2269   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7017   
all events                  plain SMLP4Rec                           0.4206   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7608   

                                                                  ndcg@10  \
seen users                  plain SMLP4Rec                         0.2852   
                            hybrid (SMLP4Rec + prior + sameDest)   0.5161   
unseen users                plain SMLP4Rec                         0.2163   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4589   
unseen, L>=1                plain SMLP4Rec                         0.2846   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4896   
unseen, L=0 (first booking) plain SMLP4Rec                         0.1118   
                            hybrid (SMLP4Rec + prior

VALID


n  \
seen users                  plain SMLP4Rec                        161151   
                            hybrid (SMLP4Rec + prior + sameDest)  161151   
unseen users                plain SMLP4Rec                        108322   
                            hybrid (SMLP4Rec + prior + sameDest)  108322   
unseen, L>=1                plain SMLP4Rec                         57519   
                            hybrid (SMLP4Rec + prior + sameDest)   57519   
unseen, L=0 (first booking) plain SMLP4Rec                         50803   
                            hybrid (SMLP4Rec + prior + sameDest)   50803   
all events                  plain SMLP4Rec                        269473   
                            hybrid (SMLP4Rec + prior + sameDest)  269473   

                                                                  recall@5  \
seen users                  plain SMLP4Rec                          0.3433   
                            hybrid (SMLP4Rec + prior + sameDest)    0.6277   
unseen users                plain SMLP4Rec                          0.2245   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5620   
unseen, L>=1                plain SMLP4Rec                          0.3123   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5912   
unseen, L=0 (first booking) plain SMLP4Rec                          0.1251   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5289   
all events                  plain SMLP4Rec                          0.2956   
                            hybrid (SMLP4Rec + prior + sameDest)    0.6013   

                                                                  ndcg@5  \
seen users                  plain SMLP4Rec                        0.2455   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4671   
unseen users                plain SMLP4Rec                        0.1627   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4015   
unseen, L>=1                plain SMLP4Rec                        0.2358   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4369   
unseen, L=0 (first booking) plain SMLP4Rec                        0.0800   
                            hybrid (SMLP4Rec + prior + sameDest)  0.3614   
all events                  plain SMLP4Rec                        0.2122   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4407   

                                                                  recall@10  \
seen users                  plain SMLP4Rec                           0.4740   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7912   
unseen users                plain SMLP4Rec                           0.3265   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7409   
unseen, L>=1                plain SMLP4Rec                           0.4293   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7631   
unseen, L=0 (first booking) plain SMLP4Rec                           0.2101   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7158   
all events                  plain SMLP4Rec                           0.4147   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7710   

                                                                  ndcg@10  \
seen users                  plain SMLP4Rec                         0.2875   
                            hybrid (SMLP4Rec + prior + sameDest)   0.5201   
unseen users                plain SMLP4Rec                         0.1954   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4594   
unseen, L>=1                plain SMLP4Rec                         0.2733   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4925   
unseen, L=0 (first booking) plain SMLP4Rec                         0.1073   
                            hybrid (SMLP4Rec + prior

In [8]:
boot = {
    subset: paired_bootstrap(sub["test"][HYB][subset], sub["test"][PLAIN][subset], k=5)
    for subset in SUBSETS
}
print("TEST, hybrid minus plain SMLP4Rec, paired bootstrap (1000 resamples, seed 0, 95% CI)")
pd.DataFrame(
    {
        s: {m: f"{v[m]['diff']:+.4f} [{v[m]['ci95'][0]:+.4f}, {v[m]['ci95'][1]:+.4f}]" for m in ("recall@5", "ndcg@5")}
        for s, v in boot.items()
    }
).T

TEST, hybrid minus plain SMLP4Rec, paired bootstrap (1000 resamples, seed 0, 95% CI)


,recall@5,ndcg@5
seen users,"+0.2822 [+0.2795, +0.2851]","+0.2208 [+0.2189, +0.2228]"
unseen users,"+0.3062 [+0.3033, +0.3092]","+0.2211 [+0.2191, +0.2233]"
"unseen, L>=1","+0.2577 [+0.2542, +0.2617]","+0.1897 [+0.1871, +0.1923]"
"unseen, L=0 (first booking)","+0.3804 [+0.3752, +0.3855]","+0.2692 [+0.2655, +0.2729]"
all events,"+0.2934 [+0.2912, +0.2955]","+0.2210 [+0.2194, +0.2224]"


## 6. Save

`results/week3_implementation/smlprec_expedia_seen_unseen_users.json` (and a copy in the run folder).

In [9]:
result = {
    "run": "seen vs unseen users: plain SMLP4Rec vs hybrid (SMLP4Rec + prior + sameDest), fixed checkpoint and weights, no retraining",
    "dataset": "expedia (train.csv, is_booking == 1; item = hotel_cluster)",
    "checkpoint": str(RUN_DIR / "best.pth"),
    "definitions": {
        "seen_user": "user has >= 1 row in the RecBole train split (a train target)",
        "unseen_user": "no train row; events = warm rows (L >= 1) of such users + first bookings (L = 0) in the window",
        "plain_SMLP4Rec": "model softmax on L >= 1; global popularity on L = 0",
        "hybrid": "notebook 04 formula and weights on L >= 1; destination prior only on L = 0",
    },
    "hybrid_weights": {"alpha": ALPHA, "scheme": SCHEME, "final_weights_per_cell": {str(k): list(v) for k, v in FINAL_W.items()}},
    "presence": presence,
    "results": results,
    "bootstrap_test_95ci_hybrid_minus_plain": boot,
}
for path in (RUN_DIR / "seen_unseen_users.json", REF_DIR / "smlprec_expedia_seen_unseen_users.json"):
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("saved", REF_DIR / "smlprec_expedia_seen_unseen_users.json")

saved C:\workspace\PROJECTS\next-best-product-recommendation\results\week3_implementation\smlprec_expedia_seen_unseen_users.json
